In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror{font-family:Consolas; font-size:15pt;}
div.output{font-size:12pt; font-weight:bold;}
div.input{font-family:Consolas; font-size:12pt;}
div.prompt{min-width:70px;}
div#toc-wrapper {padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

In [20]:
import numpy as np
import pandas as pd
from tensorflow.keras.datasets import mnist # mnist 훈련셋과 테스트셋
from tensorflow.keras.utils import to_categorical # 원핫인코딩
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from matplotlib import pyplot as plt # 학습과정 loss와 acc 시각화
# quiz에서는 scale조정, train_test_split 등을 추가
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split # 데이터 분리

- Red Wine 등급 예측
1. 데이터 셋 확보 및 전처리
    csv -> 결측치 처리 -> 독립변수와 타겟변수 분리 -> 독립변수 스케일조정, 
    -> 타겟변수의 원핫인코딩 -> 훈련셋과 테스트셋 분리(train_test_split이용해서 층화추출)
2. 모델 구성(입력11, 출력9-to_categorical | 출력6-pd.get_dummies.)
    layer층은 4개까지만 쌓기
3. 모델 학습 과정 설정
4. 모델 학습(callbacks 이용)
5. 모델 평가(그래프, 평가, 교차표)
6. 모델 저장 & 사용

# 1. 데이터 확보 & 전처리

In [12]:
# 데이터 읽어오기
redwine = pd.read_csv('data/winequality-red.csv', sep=';')
redwine.shape

(1599, 12)

In [7]:
# 결측치있는지 확인하기
redwine.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [8]:
# 0~2, 9~ 등급은 알아맞추기 어려움, 5 6등급은 잘맞출거같음
# 3등급도 잘 맞추게 하려면 전체를 다 10으로 내리기(근데 10은 너무 없어서 X)
redwine['quality'].value_counts() 

5    681
6    638
7    199
4     53
8     18
3     10
Name: quality, dtype: int64

In [66]:
X_data = redwine.iloc[:, :-1].values
y_data = redwine.iloc[:, -1].values
X_data.shape, y_data.shape

((1599, 11), (1599,))

In [67]:
scaler_X = MinMaxScaler()
scaled_X_data = scaler_X.fit_transform(X_data)
scaled_X_data[:2]

array([[0.24778761, 0.39726027, 0.        , 0.06849315, 0.10684474,
        0.14084507, 0.09893993, 0.56754772, 0.60629921, 0.13772455,
        0.15384615],
       [0.28318584, 0.52054795, 0.        , 0.11643836, 0.14357262,
        0.33802817, 0.2155477 , 0.49412628, 0.36220472, 0.20958084,
        0.21538462]])

In [75]:
X_train, X_test, y_train, y_test = train_test_split(scaled_X_data,
                                                    y_data,
                                                    test_size=0.3, # 테스트셋 비율
                                                    random_state=7,
                                                    stratify=y_data) # 층화추출
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((1119, 11), (1119,), (480, 11), (480,))

In [77]:
Y_train = to_categorical(y_train)
Y_test = to_categorical(y_test)
Y_train.shape, Y_test.shape

((1119, 9), (480, 9))

# 2. 모델 구성

In [55]:
model = Sequential()
model.add(Input(shape=(11,)))
model.add(Dense(units=16, activation='relu'))
model.add(Dense(units=32, activation='relu'))
model.add(Dense(units=9, activation='softmax'))
model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_6 (Dense)             (None, 16)                192       
                                                                 
 dense_7 (Dense)             (None, 32)                544       
                                                                 
 dense_8 (Dense)             (None, 9)                 297       
                                                                 
Total params: 1,033
Trainable params: 1,033
Non-trainable params: 0
_________________________________________________________________


# 3. 모델 학습과정 설정

In [56]:
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

# 4. 모델 학습